# Browser agent (multi-turn) — Browserless × LlamaIndex (Python)

**Tier 2.** Multi-turn `browserless_agent` workflow: navigate → snapshot → click → re-snapshot → extract. Each step is a separate `tools/call` request, so they must share the same MCP session and land on the same droplet.

## Important: LlamaIndex multi-turn pattern

`llama-index-tools-mcp`'s default flow opens a **fresh** MCP `ClientSession` for every tool call — `BasicMCPClient.call_tool()` runs `_run_session()` internally per invocation, with a fresh `Mcp-Session-Id`. That works for stateless tools (smartscraper, search, etc.) but breaks multi-turn `browserless_agent` flows: each call gets routed to a different browser, so call 2's `snapshot` returns `about:blank`.

To preserve state, use the `client._run_session()` async context manager directly and call `session.call_tool(...)` inside it. That keeps a single MCP session alive across all calls. The method is underscore-prefixed — Python convention for private — but it's just an `@asynccontextmanager`. Until upstream ships a public `client.session()` API, this is the cleanest workaround.

Also: pass `timeout=120` to `BasicMCPClient`. The default 30s is too short for the first call's MCP handshake + browser warm-up.

In [ ]:
%pip install -q llama-index llama-index-tools-mcp llama-index-llms-anthropic

In [ ]:
import os
from llama_index.tools.mcp import BasicMCPClient

MCP_URL = os.environ.get("BROWSERLESS_MCP_URL", "https://mcp.browserless.io/mcp")

client = BasicMCPClient(
    MCP_URL,
    headers={"Authorization": f"Bearer {os.environ['BROWSERLESS_TOKEN']}"},
    timeout=120,
)

## Direct multi-turn proof

Three sequential `browserless_agent` calls inside a single MCP session: `goto` → `snapshot` → `text`. Call 3 must return `"Example Domain"` — proof that the navigation from call 1 persisted into call 3.

In [ ]:
async with client._run_session() as session:
    listed = await session.list_tools()
    print(f"Loaded {len(listed.tools)} tools")

    await session.call_tool(
        "browserless_agent",
        arguments={"method": "goto", "params": {"url": "https://example.com"}},
    )
    await session.call_tool("browserless_agent", arguments={"method": "snapshot"})
    result = await session.call_tool(
        "browserless_agent",
        arguments={"method": "text", "params": {"selector": "h1"}},
    )
    text = result.content[0].text
    print(text)
    assert "Example Domain" in text, "Session state was not preserved between calls"

## Full agent loop (LLM-driven)

Wrap the session-bound tool calls in a thin function tool the agent can invoke through the LLM. The whole agent run lives inside a single `client._run_session()` block so every browser action shares one MCP session.

In [ ]:
from llama_index.core.tools import FunctionTool
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.anthropic import Anthropic

async with client._run_session() as session:
    async def browser_action(method: str, params: dict | None = None) -> str:
        """Drive the Browserless multi-turn browser agent. Methods include `goto`, `snapshot`, `click`, `type`, `text`, `evaluate`. Pass `params` per the method's schema (e.g. {\"url\": \"...\"} for goto, {\"selector\": \"...\"} for text)."""
        result = await session.call_tool(
            "browserless_agent",
            arguments={"method": method, "params": params or {}},
        )
        return result.content[0].text

    browser_tool = FunctionTool.from_defaults(
        async_fn=browser_action,
        name="browserless_agent",
        description="Drive a stateful browser session. Use methods like goto, snapshot, click, type, text, evaluate. Snapshot before extracting elements.",
    )

    agent = FunctionAgent(
        name="BrowserlessBrowserAgent",
        description="Drives a real browser to navigate, click, and extract content.",
        llm=Anthropic(model="claude-sonnet-4-6"),
        tools=[browser_tool],
        system_prompt=(
            "You drive a real browser via the browserless_agent tool. "
            "Always snapshot the page before extracting elements so you know what selectors are available. "
            "Use selector references from the snapshot when calling text/click."
        ),
    )

    response = await agent.run(
        "Navigate to https://news.ycombinator.com, click the first story link, and report the title and first paragraph of that page."
    )
    print(response)